# 01 — Exploratory analysis of the IBM Telco churn data

**Goal:** understand the raw file, its quirks, and where churn concentrates before any modelling.

* 7,043 customers × 21 columns, one row per customer (point-in-time snapshot).
* Target: `Churn` (Yes/No). Base rate ≈ 26–27 %.
* Known quirk: `TotalCharges` is text and has blank cells for tenure-0 customers.

> Run `python data/download_data.py` first for the real file; otherwise this notebook falls back to the synthetic fixture.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from telco_churn import config, ingest, clean, features, quality, model, warehouse

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", palette=["#117865", "#C0392B", "#1F2A44", "#8A94A6"])

# Real file if present, otherwise the synthetic fixture
RAW = config.DEFAULT_RAW_FILE if config.DEFAULT_RAW_FILE.exists() else config.FIXTURE_RAW_FILE
print("Using:", RAW.name, "| synthetic fixture" if "fixture" in RAW.name else "| real IBM data")

## 1. Raw file — types and quirks

In [ ]:
raw = ingest.load_raw_telco(RAW)
display(raw.head())
print(raw.shape)
print("Blank TotalCharges:", (raw["TotalCharges"].str.strip() == "").sum(),
      "| all with tenure 0?", (raw.loc[raw["TotalCharges"].str.strip() == "", "tenure"] == "0").all())

## 2. Clean + conform

In [ ]:
df, report = clean.clean_telco(raw)
print(report.as_dict())
df.describe(include="all").T.head(25)

## 3. Where does churn concentrate?

In [ ]:
def churn_by(col):
    g = df.groupby(col)["churn_flag"].agg(customers="size", churn_rate="mean").sort_values("churn_rate", ascending=False)
    g["churn_rate"] = (g["churn_rate"] * 100).round(1)
    return g

for col in ["contract", "internet_service", "payment_method", "tech_support", "senior_citizen"]:
    display(churn_by(col))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["contract", "internet_service", "payment_method"]):
    g = churn_by(col).reset_index()
    sns.barplot(data=g, x=col, y="churn_rate", ax=ax, color="#117865")
    ax.set_title(f"Churn rate by {col}"); ax.set_ylabel("%"); ax.tick_params(axis="x", rotation=20)
plt.tight_layout()

## 4. Tenure and charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x="tenure_months", hue="churn", bins=36, multiple="stack", ax=axes[0])
axes[0].set_title("Tenure distribution by churn")
sns.kdeplot(data=df, x="monthly_charges", hue="churn", fill=True, common_norm=False, ax=axes[1])
axes[1].set_title("Monthly charges by churn")
plt.tight_layout()

## 5. Take-aways to carry into modelling
1. Contract type and tenure dominate — month-to-month customers in their first year are the risk pool.
2. Fiber-optic customers churn far more than DSL — likely price/expectation, not the technology itself.
3. Electronic-check payers churn ~2–3× more than automatic payers — a friction/engagement proxy.
4. Protective add-ons (tech support, online security) are associated with lower churn.